# Convert VAE embeddings to MoPaDi H5 format

This notebook converts the CSV embeddings (Patient_ID + 1024 latent dimensions) into the H5 format expected by MoPaDi's MIL classifier.

**MoPaDi requirements:**
- Each patient has one H5 file with a `feats` dataset shaped `[N_instances, feature_dim]`
- Files organized into train/test folders
- Clinical table CSV with `PATIENT` column matching file names
- Feature dimension must match config (1024 in our case)


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import h5py
from sklearn.model_selection import train_test_split

# Paths
encoded_csv_path = Path("../../vae_output/brca_gene_expression_with_subtypes_vae_encoded_data.csv")
metadata_csv_path = Path("../../data/brca_gene_expression_with_subtypes.csv")
output_base_dir = Path("../../mopadi_features")

# Output directories
feat_train_dir = output_base_dir / "train"
feat_test_dir = output_base_dir / "test"
clini_table_path = output_base_dir / "clinical_table.csv"

# Create output directories
feat_train_dir.mkdir(parents=True, exist_ok=True)
feat_test_dir.mkdir(parents=True, exist_ok=True)

print("Directories created:", feat_train_dir, feat_test_dir)


Directories created: ../../mopadi_features/train ../../mopadi_features/test


In [ ]:
# Load embeddings CSV
df_embeddings = pd.read_csv(encoded_csv_path, index_col="Patient_ID")
print(f"Loaded embeddings: {df_embeddings.shape}")
print(f"Feature columns: {df_embeddings.columns[:5].tolist()}... (total {len(df_embeddings.columns)} dims)")

# Load metadata for labels
df_metadata = pd.read_csv(metadata_csv_path)
if "Patient_ID" in df_metadata.columns:
    df_metadata = df_metadata.set_index("Patient_ID")
print(f"\nLoaded metadata: {df_metadata.shape}")
print(f"Metadata columns: {df_metadata.columns.tolist()}")


In [3]:
# Align embeddings with metadata
common_patients = df_embeddings.index.intersection(df_metadata.index)
df_embeddings_aligned = df_embeddings.loc[common_patients]
df_metadata_aligned = df_metadata.loc[common_patients]

print(f"Aligned {len(common_patients)} patients")
print(f"Embeddings shape: {df_embeddings_aligned.shape}")

# Identify label column (assuming Majority_Subtype_mRNA or similar)
label_candidates = ["Majority_Subtype_mRNA", "Subtype", "label", "TARGET"]
label_col = next((col for col in label_candidates if col in df_metadata_aligned.columns), None)
if label_col is None:
    print(f"\nWarning: No label column found. Available columns: {df_metadata_aligned.columns.tolist()}")
    print("You may need to manually specify the label column.")
else:
    print(f"\nUsing label column: {label_col}")
    print(f"Label distribution:\n{df_metadata_aligned[label_col].value_counts()}")


Aligned 981 patients
Embeddings shape: (996, 1024)

Using label column: Majority_Subtype_mRNA
Label distribution:
Majority_Subtype_mRNA
LumA      514
LumB      195
Basal     175
Her2       74
Normal     38
Name: count, dtype: int64


In [4]:
# Split into train/test (80/20 split, stratified if labels available)
test_size = 0.2
random_state = 42

if label_col and df_metadata_aligned[label_col].notna().all():
    train_idx, test_idx = train_test_split(
        df_embeddings_aligned.index,
        test_size=test_size,
        random_state=random_state,
        stratify=df_metadata_aligned[label_col]
    )
    print(f"Stratified split: {len(train_idx)} train, {len(test_idx)} test")
else:
    train_idx, test_idx = train_test_split(
        df_embeddings_aligned.index,
        test_size=test_size,
        random_state=random_state
    )
    print(f"Random split: {len(train_idx)} train, {len(test_idx)} test")

train_patients = sorted(train_idx)
test_patients = sorted(test_idx)


Stratified split: 796 train, 200 test


In [5]:
def save_patient_h5(patient_id, embedding_array, output_dir):
    """
    Save a single patient's embedding as H5 file.
    
    Args:
        patient_id: Patient identifier (used as filename)
        embedding_array: numpy array of shape (feature_dim,) or (N_instances, feature_dim)
        output_dir: Directory to save the H5 file
    """
    # Ensure embedding is 2D: (N_instances, feature_dim)
    if embedding_array.ndim == 1:
        embedding_array = embedding_array.reshape(1, -1)
    
    h5_path = output_dir / f"{patient_id}.h5"
    with h5py.File(h5_path, 'w') as f:
        f.create_dataset('feats', data=embedding_array, dtype='float32')
    
    return h5_path

# Test with one patient
test_patient = train_patients[0]
test_embedding = df_embeddings_aligned.loc[test_patient].values
test_path = save_patient_h5(test_patient, test_embedding, feat_train_dir)

# Verify the H5 file structure
with h5py.File(test_path, 'r') as f:
    print(f"Test H5 file structure:")
    print(f"  Keys: {list(f.keys())}")
    print(f"  'feats' shape: {f['feats'].shape}")
    print(f"  'feats' dtype: {f['feats'].dtype}")
    
print(f"\n✓ H5 format verified. Ready to convert all patients.")


Test H5 file structure:
  Keys: ['feats']
  'feats' shape: (1, 1024)
  'feats' dtype: float32

✓ H5 format verified. Ready to convert all patients.


In [6]:
# Convert train patients to H5 files
print("Converting train patients to H5...")
train_h5_paths = []
for patient_id in train_patients:
    embedding = df_embeddings_aligned.loc[patient_id].values
    h5_path = save_patient_h5(patient_id, embedding, feat_train_dir)
    train_h5_paths.append(h5_path)

print(f"✓ Created {len(train_h5_paths)} train H5 files in {feat_train_dir}")


Converting train patients to H5...
✓ Created 796 train H5 files in ../../mopadi_features/train


In [7]:
# Convert test patients to H5 files
print("Converting test patients to H5...")
test_h5_paths = []
for patient_id in test_patients:
    embedding = df_embeddings_aligned.loc[patient_id].values
    h5_path = save_patient_h5(patient_id, embedding, feat_test_dir)
    test_h5_paths.append(h5_path)

print(f"✓ Created {len(test_h5_paths)} test H5 files in {feat_test_dir}")


Converting test patients to H5...
✓ Created 200 test H5 files in ../../mopadi_features/test


In [8]:
# Create clinical table CSV for MoPaDi
# MoPaDi expects: PATIENT column matching H5 filenames, and target label column
all_patients = train_patients + test_patients
df_clinical = pd.DataFrame(index=all_patients)
df_clinical.index.name = "PATIENT"
df_clinical = df_clinical.reset_index()

# Add label column if available
if label_col:
    df_clinical[label_col] = df_clinical["PATIENT"].map(df_metadata_aligned[label_col])
    print(f"Added label column: {label_col}")
    print(f"Label distribution:\n{df_clinical[label_col].value_counts()}")
else:
    print("Warning: No label column added. You may need to add it manually.")

# Add split information
df_clinical["split"] = df_clinical["PATIENT"].apply(
    lambda x: "train" if x in train_patients else "test"
)

# Save clinical table
df_clinical.to_csv(clini_table_path, index=False)
print(f"\n✓ Saved clinical table to {clini_table_path}")
print(f"\nClinical table preview:")
print(df_clinical.head())


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [9]:
print(df_metadata_aligned.index.duplicated().any())

True


In [10]:
print(df_metadata_aligned.index.value_counts()[df_metadata_aligned.index.value_counts() > 1])

Patient_ID
TCGA-A7-A26J    3
TCGA-A7-A0DB    3
TCGA-A7-A13D    3
TCGA-A7-A13E    3
TCGA-A7-A26E    3
TCGA-A7-A13G    2
TCGA-AC-A2QH    2
TCGA-A7-A26F    2
TCGA-A7-A26I    2
TCGA-AC-A3OD    2
Name: count, dtype: int64


## Summary and MoPaDi Configuration

**Generated files:**
- Train H5 files: `../../mopadi_features/train/*.h5` (each contains `feats` dataset with shape `[1, 1024]`)
- Test H5 files: `../../mopadi_features/test/*.h5`
- Clinical table: `../../mopadi_features/clinical_table.csv`

**MoPaDi config.yaml settings:**

Add/modify these in your MoPaDi `conf.yaml`:

```yaml
mil_classifier:
  feat_path_train: "../../mopadi_features/train"
  feat_path_test: "../../mopadi_features/test"
  clini_table: "../../mopadi_features/clinical_table.csv"
  target_label: "Majority_Subtype_mRNA"  # or your label column name
  target_dict:  # Map string labels to integers if needed
    LumA: 0
    Basal: 1
    # ... add other subtypes
  nr_feats: 1  # Number of instances per patient (we have 1 embedding per patient)
  fname_index: 0  # Patient ID is the full filename (before .h5)
  
# MIL model config
mil_config:
  dim: 1024  # Must match your embedding dimension!
```

**Next steps:**
1. Update MoPaDi's `conf.yaml` with the paths above
2. Run: `mopadi mil --config conf.yaml --mode train` (or `crossval`)
